In [0]:
from pyspark.sql import functions as F

# 1. Read the Silver Delta from S3
silver_path = "s3a://world-bank-api/worldbank/silver/dim_country"

silver_df = (
    spark.read
    .format("delta")
    .load(silver_path)
)

display(silver_df.limit(10))
print("Silver row count:", silver_df.count())
silver_df.printSchema()


country_code,iso2_code,country_name,region_code,region_name,admin_region_code,admin_region_name,income_level_code,income_level,lending_type_code,lending_type,capital_city,longitude,latitude,ingestion_timestamp,is_aggregate,is_mappable
ABW,AW,Aruba,LCN,Latin America & Caribbean,,,HIC,High Income,LNX,Not Classified,Oranjestad,-70.0167,12.5167,2026-02-20T21:28:12.197Z,false,true
AFE,ZH,Africa Eastern And Southern,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
AFG,AF,Afghanistan,MEA,"Middle East, North Africa, Afghanistan & Pakistan",MNA,"Middle East, North Africa, Afghanistan & Pakistan (excluding High Income)",LIC,Low Income,IDX,Ida,Kabul,69.1761,34.5228,2026-02-20T21:28:12.197Z,false,true
AFR,A9,Africa,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
AFW,ZI,Africa Western And Central,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
AGO,AO,Angola,SSF,Sub-saharan Africa,SSA,Sub-saharan Africa (excluding High Income),LMC,Lower Middle Income,IBD,Ibrd,Luanda,13.242,-8.81155,2026-02-20T21:28:12.197Z,false,true
ALB,AL,Albania,ECS,Europe & Central Asia,ECA,Europe & Central Asia (excluding High Income),UMC,Upper Middle Income,IBD,Ibrd,Tirane,19.8172,41.3317,2026-02-20T21:28:12.197Z,false,true
AND,AD,Andorra,ECS,Europe & Central Asia,,,HIC,High Income,LNX,Not Classified,Andorra La Vella,1.5218,42.5075,2026-02-20T21:28:12.197Z,false,true
ARB,1A,Arab World,NA,Aggregates,,,NA,Aggregates,,Aggregates,null,null,null,2026-02-20T21:28:12.197Z,true,false
ARE,AE,United Arab Emirates,MEA,"Middle East, North Africa, Afghanistan & Pakistan",,,HIC,High Income,LNX,Not Classified,Abu Dhabi,54.3705,24.4764,2026-02-20T21:28:12.197Z,false,true


Silver row count: 296
root
 |-- country_code: string (nullable = true)
 |-- iso2_code: string (nullable = true)
 |-- country_name: string (nullable = true)
 |-- region_code: string (nullable = true)
 |-- region_name: string (nullable = true)
 |-- admin_region_code: string (nullable = true)
 |-- admin_region_name: string (nullable = true)
 |-- income_level_code: string (nullable = true)
 |-- income_level: string (nullable = true)
 |-- lending_type_code: string (nullable = true)
 |-- lending_type: string (nullable = true)
 |-- capital_city: string (nullable = true)
 |-- longitude: double (nullable = true)
 |-- latitude: double (nullable = true)
 |-- ingestion_timestamp: timestamp (nullable = true)
 |-- is_aggregate: boolean (nullable = true)
 |-- is_mappable: boolean (nullable = true)



In [0]:
# Keep only real countries
dim_country_df = (
    silver_df
    .filter(F.col("is_aggregate") == False)   # or == 0 depending on your type
    .select(
        "country_code",
        "iso2_code",
        "country_name",
        "capital_city",
        "region_code",
        "admin_region_code",
        "income_level_code",
        "lending_type_code",
        "longitude",
        "latitude",
        "is_mappable"
    )
)

# Add a surrogate key (for DW usage later)
dim_country_df = dim_country_df.withColumn(
    "country_sk",
    F.monotonically_increasing_id()
)

# Reorder columns nicely
dim_country_df = dim_country_df.select(
    "country_sk",
    "country_code",
    "iso2_code",
    "country_name",
    "capital_city",
    "region_code",
    "admin_region_code",
    "income_level_code",
    "lending_type_code",
    "longitude",
    "latitude",
    "is_mappable"
)

display(dim_country_df.limit(10))
print("dim_country rows:", dim_country_df.count())


country_sk,country_code,iso2_code,country_name,capital_city,region_code,admin_region_code,income_level_code,lending_type_code,longitude,latitude,is_mappable
0,ABW,AW,Aruba,Oranjestad,LCN,,HIC,LNX,-70.0167,12.5167,true
1,AFG,AF,Afghanistan,Kabul,MEA,MNA,LIC,IDX,69.1761,34.5228,true
2,AGO,AO,Angola,Luanda,SSF,SSA,LMC,IBD,13.242,-8.81155,true
3,ALB,AL,Albania,Tirane,ECS,ECA,UMC,IBD,19.8172,41.3317,true
4,AND,AD,Andorra,Andorra La Vella,ECS,,HIC,LNX,1.5218,42.5075,true
5,ARE,AE,United Arab Emirates,Abu Dhabi,MEA,,HIC,LNX,54.3705,24.4764,true
6,ARG,AR,Argentina,Buenos Aires,LCN,LAC,UMC,IBD,-58.4173,-34.6118,true
7,ARM,AM,Armenia,Yerevan,ECS,ECA,UMC,IBD,44.509,40.1596,true
8,ASM,AS,American Samoa,Pago Pago,EAS,,HIC,LNX,-170.691,-14.2846,true
9,ATG,AG,Antigua And Barbuda,Saint John's,LCN,,HIC,IBD,-61.8456,17.1175,true


dim_country rows: 217


In [0]:
dim_region_df = (
    silver_df
    .select("region_code", "region_name")
    .where(F.col("region_code").isNotNull())
    .dropDuplicates(["region_code"])
    .withColumn("region_sk", F.monotonically_increasing_id())
    .select("region_sk", "region_code", "region_name")
    .orderBy("region_code")
)

display(dim_region_df)
print("dim_region rows:", dim_region_df.count())


region_sk,region_code,region_name
3,EAS,East Asia & Pacific
2,ECS,Europe & Central Asia
5,LCN,Latin America & Caribbean
6,MEA,"Middle East, North Africa, Afghanistan & Pakistan"
1,NA,Aggregates
4,NAC,North America
0,SAS,South Asia
7,SSF,Sub-saharan Africa


dim_region rows: 8


In [0]:
dim_admin_region_df = (
    silver_df
    .select("admin_region_code", "admin_region_name")
    .where(F.col("admin_region_code").isNotNull())
    .dropDuplicates(["admin_region_code"])
    .withColumn("admin_region_sk", F.monotonically_increasing_id())
    .select("admin_region_sk", "admin_region_code", "admin_region_name")
    .orderBy("admin_region_code")
)

display(dim_admin_region_df)
print("dim_admin_region rows:", dim_admin_region_df.count())


admin_region_sk,admin_region_code,admin_region_name
4,,
6,EAP,East Asia & Pacific (excluding High Income)
1,ECA,Europe & Central Asia (excluding High Income)
3,LAC,Latin America & Caribbean (excluding High Income)
0,MNA,"Middle East, North Africa, Afghanistan & Pakistan (excluding High Income)"
2,SAS,South Asia
5,SSA,Sub-saharan Africa (excluding High Income)


dim_admin_region rows: 7


In [0]:
dim_income_level_df = (
    silver_df
    .select("income_level_code", "income_level")
    .where(F.col("income_level_code").isNotNull())
    .dropDuplicates(["income_level_code"])
    .withColumn("income_level_sk", F.monotonically_increasing_id())
    .select("income_level_sk", "income_level_code", "income_level")
    .orderBy("income_level_code")
)

display(dim_income_level_df)
print("dim_income_level rows:", dim_income_level_df.count())


income_level_sk,income_level_code,income_level
2,HIC,High Income
3,INX,Not Classified
5,LIC,Low Income
4,LMC,Lower Middle Income
0,NA,Aggregates
1,UMC,Upper Middle Income


dim_income_level rows: 6


In [0]:
dim_lending_type_df = (
    silver_df
    .select("lending_type_code", "lending_type")
    .where(F.col("lending_type_code").isNotNull())
    .dropDuplicates(["lending_type_code"])
    .withColumn("lending_type_sk", F.monotonically_increasing_id())
    .select("lending_type_sk", "lending_type_code", "lending_type")
    .orderBy("lending_type_code")
)

display(dim_lending_type_df)
print("dim_lending_type rows:", dim_lending_type_df.count())


lending_type_sk,lending_type_code,lending_type
4,,Aggregates
0,IBD,Ibrd
1,IDB,Blend
2,IDX,Ida
3,LNX,Not Classified


dim_lending_type rows: 5


In [0]:
dim_aggregate_entity_df = (
    silver_df
    .filter(F.col("is_aggregate") == True)
    .select(
        "country_code",      # acts like a group code
        "country_name",      # name like "High income"
        "region_code",
        "region_name",
        "income_level_code",
        "income_level"
    )
    .withColumn("aggregate_sk", F.monotonically_increasing_id())
    .select(
        "aggregate_sk",
        "country_code",
        "country_name",
        "region_code",
        "region_name",
        "income_level_code",
        "income_level"
    )
)

display(dim_aggregate_entity_df.limit(20))
print("dim_aggregate_entity rows:", dim_aggregate_entity_df.count())


aggregate_sk,country_code,country_name,region_code,region_name,income_level_code,income_level
0,AFE,Africa Eastern And Southern,NA,Aggregates,NA,Aggregates
1,AFR,Africa,NA,Aggregates,NA,Aggregates
2,AFW,Africa Western And Central,NA,Aggregates,NA,Aggregates
3,ARB,Arab World,NA,Aggregates,NA,Aggregates
4,BEA,East Asia & Pacific (ibrd-only Countries),NA,Aggregates,NA,Aggregates
5,BEC,Europe & Central Asia (ibrd-only Countries),NA,Aggregates,NA,Aggregates
6,BHI,Ibrd Countries Classified As High Income,NA,Aggregates,NA,Aggregates
7,BLA,Latin America & The Caribbean (ibrd-only Countries),NA,Aggregates,NA,Aggregates
8,BMN,"Middle East, North Africa, Afghanistan & Pakistan (ibrd Only)",NA,Aggregates,NA,Aggregates
9,BSS,Sub-saharan Africa (ibrd-only Countries),NA,Aggregates,NA,Aggregates


dim_aggregate_entity rows: 79


In [0]:
# Join dim_country with the code-based columns (already there), keep relationships
fact_country_membership_df = dim_country_df.select(
    "country_sk",
    "country_code",
    "region_code",
    "admin_region_code",
    "income_level_code",
    "lending_type_code",
    "is_mappable"
)

display(fact_country_membership_df)
print("fact_country_membership rows:", fact_country_membership_df.count())


country_sk,country_code,region_code,admin_region_code,income_level_code,lending_type_code,is_mappable
0,ABW,LCN,,HIC,LNX,true
1,AFG,MEA,MNA,LIC,IDX,true
2,AGO,SSF,SSA,LMC,IBD,true
3,ALB,ECS,ECA,UMC,IBD,true
4,AND,ECS,,HIC,LNX,true
5,ARE,MEA,,HIC,LNX,true
6,ARG,LCN,LAC,UMC,IBD,true
7,ARM,ECS,ECA,UMC,IBD,true
8,ASM,EAS,,HIC,LNX,true
9,ATG,LCN,,HIC,IBD,true


fact_country_membership rows: 217


In [0]:
gold_base_path = "s3a://world-bank-api/worldbank/gold"

dim_country_df.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_country")
dim_region_df.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_region")
dim_admin_region_df.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_admin_region")
dim_income_level_df.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_income_level")
dim_lending_type_df.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_lending_type")
dim_aggregate_entity_df.write.format("delta").mode("overwrite").save(f"{gold_base_path}/dim_aggregate_entity")
fact_country_membership_df.write.format("delta").mode("overwrite").save(f"{gold_base_path}/fact_country_membership")

print("✅ All dim & fact tables written to Gold S3 layer:", gold_base_path)


✅ All dim & fact tables written to Gold S3 layer: s3a://world-bank-api/worldbank/gold
